<a href="https://colab.research.google.com/github/Tulasisree/AI_NOTES/blob/main/Deep_Convolutional_Q_Learning_for_Pac_Man.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Convolutional Q-Learning for Pac-Man

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [ ]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!pip install ale-py
!apt-get install -y swig
!pip install gymnasium[box2d]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 53 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 0s (9,972 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubu

### Importing the libraries

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
from torch.utils.data import DataLoader, TensorDataset

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [ ]:
class Network(nn.Module):
  #we dont give state_size as arg since we use images now
  def __init__(self, action_size, seed = 42):
    #activate inheritance from network module
    super(Network, self).__init__()
    self.seed = torch.manual_seed(seed)
    #Building eyes
    #arg as 3 i/p channels cooresponding to RGB colors of the image
    #o/p channels 64, kernel size 8, stride 4
    self.conv1 = nn.Conv2d(3, 32, kernel_size = 8, stride = 4)
    #batch normalization layer for 2D i/p -> helps in faster and more stable training
    #no. of featuers -> no. of channels/feature maps in previous layer -> 32
    self.bn1 = nn.BatchNorm2d(32)
    # we increase the number of channels decreasing the spacial dimensions *kernel,stride)
    self.conv2 = nn.Conv2d(32, 64, kernel_size = 4, stride = 2)
    self.bn2 = nn.BatchNorm2d(64)
    self.conv3 = nn.Conv2d(64, 64, kernel_size = 3, stride = 1)
    self.bn3 = nn.BatchNorm2d(64)
    self.conv4 = nn.Conv2d(64, 128, kernel_size = 3, stride = 1)
    self.bn4 = nn.BatchNorm2d(128)

    #building brain
    #1st arg is no. of i/p features -> no. of o/p features for flatting all the previous convolutions
    #2nd arg no of o/p features -> no. of artifical neurons from the 1st fully connected layer
    self.fc1 = nn.Linear(10 * 10 * 128, 512)
    self.fc2 = nn.Linear(512, 256)
    self.fc3 = nn.Linear(256, action_size)

  #fowrad propagate the signals from pac-man images to the eyes , eyes -> fully connected layer, fc ->body of AI(playing actions)
  def forward(self,state):
    x = F.relu(self.bn1(self.conv1(state)))
    x = F.relu(self.bn2(self.conv2(x)))
    x = F.relu(self.bn3(self.conv3(x)))
    x = F.relu(self.bn4(self.conv4(x)))
    #reshape x to get flattening layer in order to fit into fc
    #1st dimenision corresponding to batch remains same, others are flattened
    x = x.view(x.size(0),-1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    return self.fc3(x)

## Part 2 - Training the AI

### Setting up the environment

In [ ]:
import ale_py
import gymnasium as gym
env = gym.make('MsPacmanNoFrameskip-v4', full_action_space = False) # The MsPacman environment was renamed 'MsPacmanNoFrameskip-v0'
state_shape = env.observation_space.shape
state_size = env.observation_space.shape[0]
number_actions = env.action_space.n
print('State shape: ', state_shape)
print('State size: ', state_size)
print('Number of actions: ', number_actions)

State shape:  (210, 160, 3)
State size:  210
Number of actions:  9


### Initializing the hyperparameters

In [ ]:
learning_rate = 5e-4
minibatch_size = 64
discount_factor = 0.99
#no need of replay buffer for conv neural network, images too big to store
#and interpolation paramaneter for this env

### Preprocessing the frames

In [ ]:
#load images and create images
from PIL import Image
from torchvision import transforms

def preprocess_frame(frame):
  #frame is in numpy array -> convert to PIL image object
  frame = Image.fromarray(frame)
  #preprocessing to create pytorch tensor
  #args contains list of tranformations we are gonna do on the prepocess object
  preprocess = transforms.Compose([
      #we need lower and sqaure dimensions. The original one to hard to train on
      transforms.Resize((128,128)),
      #converting to pytorch tenso
      transforms.ToTensor(),
  ])
  return preprocess(frame).unsqueeze(0) #extra dimension for batch index 0

### Implementing the DCQN class

In [ ]:
class Agent():
  def __init__(self,action_size):
    self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    self.action_size = action_size
    #Q-learning
    #two networks -> local q network and target q network
    self.local_qnetwork = Network(action_size).to(self.device)
    self.target_qnetwork = Network(action_size).to(self.device)
    #optimizer
    #parameters -> weights
    self.optimizer = optim.Adam(self.local_qnetwork.parameters(),lr = learning_rate)
    #memeory of AI
    self.memory = deque(maxlen = 10000)

  #stores experiences and decide when to learn from them
  def step(self,state,action,reward,next_state,done):
    #preprocess the state to make them ready to be first added to AI's memory and then fed to nn
    state = preprocess_frame(state)
    next_state = preprocess_frame(next_state)
    self.memory.append((state,action,reward,next_state,done))
    if len(self.memory) > minibatch_size:
      experiences = random.sample(self.memory, k = minibatch_size)
      self.learn(experiences, discount_factor)

  #will select an action based on the given state in the env
  #we are doing epsilon greedy action selection policy
  def act(self, state, epsilon = 0.):
    #state is image now
    #unsqueeze(0) -> we need to add extra dimension to our state for deep learning vector which will correspond to batch,0->added at the beginning
    state = preprocess_frame(state).to(self.device)
    #forward past this state to q networks to get the action values, 1st we need to set q network to eval mode
    self.local_qnetwork.eval()
    #check to make sure we are not in trainig mode but inference mode making prediction
    with torch.no_grad():
      action_values = self.local_qnetwork(state)
    #back to training mode
    self.local_qnetwork.train()
    if random.random() > epsilon:
      #argmax take numpy data format
      return np.argmax(action_values.cpu().data.numpy())
    else:
      return random.choice(np.arange(self.action_size))

  #updates agents q values based on sampled experiences
  def learn(self,experiences,discount_factor):
    states,actions,rewards,next_states,dones = zip(*experiences)
    #form stack
    states = torch.from_numpy(np.vstack(states)).float().to(self.device)
    actions = torch.from_numpy(np.vstack(actions)).long().to(self.device)
    rewards = torch.from_numpy(np.vstack(rewards)).float().to(self.device)
    next_states = torch.from_numpy(np.vstack(next_states)).float().to(self.device)
    dones = torch.from_numpy(np.vstack(dones).astype(np.uint8)).float().to(self.device)

    next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1) #unsqueezse for batch
    q_targets = rewards + (discount_factor * next_q_targets * (1-dones))
    #current q values , gather all q values
    q_expected = self.local_qnetwork(states).gather(1,actions)
    #loss btm expected and target q values
    loss = F.mse_loss(q_expected,q_targets)
    #backpropogate the loss
    #1st we do reset the optimizer
    self.optimizer.zero_grad()
    loss.backward()
    #single optimization steps lo update the parameters of the model
    self.optimizer.step()



### Initializing the DCQN agent

In [ ]:
agent = Agent(number_actions)

### Training the DCQN agent

In [ ]:
#max no. of episodes with which we want to train our agent
number_episodes = 2000
#max no. of time steps per episode
max_t = 10000
#hyperparamenters realted to epsilonn greedy actiom selection policy
eps_start = 1.0
eps_end = 0.01
#decrement for which we are gng from 1.0 to 0.01
eps_decay = 0.995
epsilon = eps_start
#contains scores of last 100 episodes
scores_window = deque(maxlen = 100)

for episode in range(1,number_episodes+1):
  #reset env to initial state
  state, _ = env.reset()
  #initialize the score
  score = 0
  #time steps over the episode
  for t in range(max_t):
    action = agent.act(state,epsilon)
    next_state,reward,done,_,_ = env.step(action)
    #training
    agent.step(state,action,reward,next_state,done)
    state = next_state
    score += reward
    if done:
      break
  scores_window.append(score)
  epsilon = max(eps_end,eps_decay*epsilon)
  #advanced printing - overriding effect
  print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_window)), end="")
  if episode % 100 == 0:
    print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_window)))
    if np.mean(scores_window)>=500.0:
      print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(episode-100, np.mean(scores_window)))
      #save the winning model to file checkpoint.pth check files section
      torch.save(agent.local_qnetwork.state_dict(), 'checkpoint.pth')
      #break the training loop since we reached the solution
      break

Episode 15	Average Score: 178.67

## Part 3 - Visualizing the results

In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        action = agent.act(state)
        state, reward, done, _, _ = env.step(action)
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, 'MsPacmanNoFrameskip-v0') # The MsPacman environment was renamed 'MsPacmanNoFrameskip-v0'

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()